# Loan Approval Prediction — Model Training

Run this notebook in Google Colab. Upload your loan approval CSV when prompted. It trains a Random Forest pipeline and downloads `loan_model.joblib` for the Streamlit app.

In [ ]:
import pandas as pd
import joblib
import sklearn
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

print('scikit-learn version:', sklearn.__version__)

## 1. Upload the dataset
Choose your loan approval CSV file when the upload window appears.

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise ValueError('No CSV file was uploaded.')
csv_name = next(iter(uploaded))
df = pd.read_csv(csv_name)
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()
print('Dataset shape:', df.shape)
display(df.head())
print('Columns:', df.columns.tolist())

## 2. Select features and target

In [ ]:
FEATURES_NUM = [
    'no_of_dependents', 'income_annum', 'loan_amount', 'loan_term',
    'cibil_score', 'residential_assets_value', 'commercial_assets_value',
    'luxury_assets_value', 'bank_asset_value'
]
FEATURES_CAT = ['education', 'self_employed']
TARGET = 'loan_status'

required = FEATURES_NUM + FEATURES_CAT + [TARGET]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

X = df[FEATURES_NUM + FEATURES_CAT].copy()
y = (df[TARGET].str.strip().str.lower() == 'approved').astype(int)
print('Target counts (0 = rejected, 1 = approved):')
print(y.value_counts())

## 3. Train and evaluate the model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[('categorical', OneHotEncoder(handle_unknown='ignore'), FEATURES_CAT)],
    remainder='passthrough'
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=300, random_state=42))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, predictions), 4))
print(classification_report(y_test, predictions, target_names=['Rejected', 'Approved']))

## 4. Save and download the trained model
Upload this file to the root of your GitHub repository along with `app.py`.

In [ ]:
MODEL_FILE = 'loan_model.joblib'
joblib.dump(model, MODEL_FILE)
print('Saved:', MODEL_FILE)
print('scikit-learn version:', sklearn.__version__)
files.download(MODEL_FILE)